In [ ]:
import scipy.ndimage as ndi
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import math
import nibabel as nib
import skimage.transform
import pathlib
import pandas as pd
# from tqdm import tqdm
# import argparse
# import tensorflow.keras.backend as K
from scipy import ndimage
import skimage.feature, skimage.measure, skimage.morphology, skimage.segmentation
# import SimpleITK as stk
import glob
# from google.colab.patches import cv2_imshow
from sklearn.model_selection import train_test_split
import tensorflow as tf
import keras
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Flatten, BatchNormalization, \
    Conv2D, MaxPooling2D, ZeroPadding2D, Input, Embedding, \
    Lambda, UpSampling2D, Cropping2D, Conv2DTranspose, concatenate

from keras.optimizers import SGD, Adam
from keras.callbacks import ModelCheckpoint, LearningRateScheduler, ReduceLROnPlateau, CSVLogger
# from keras.preprocessing.image import ImageDataGenerator
from keras import backend as K
from keras import optimizers
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import array_to_img
from skimage.transform import resize
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split
from skimage.transform import resize
# from tensorflow.keras.utils import array_to_img, Sequence

In [ ]:
#### Load train data of 3d mask with npy
#################################################

path = "/kaggle/input/newnodules/new_ct (S123)/"
mask_path = "/kaggle/input/newnodules/new_mask (S123)/"

CT_list = glob.glob(path + "*.npy")
mask_list = glob.glob(mask_path + "*.npy")

Patient_ID = []
x_train = []
y_train = []



for i in range(len(CT_list)):
  Patient_ID.append(CT_list[i].split("/")[-1][:-4])
  print(i)

  temp = np.load(CT_list[i])
  mask_temp = np.load(mask_path + Patient_ID[i] + ".npy")

  x = np.zeros((np.shape(temp)[0], 256, 256))
  y = np.zeros((np.shape(temp)[0], 256, 256))

  for j in range(np.shape(temp)[0]):



    x[j, :, :] = (resize(temp[j, :, :], (256, 256)))
    y[j, :, :] = (resize(mask_temp[j, :, :], (256, 256)))



  x_train.append(x)
  y_train.append(y)



x_train = np.concatenate(x_train, axis=0)
y_train = np.concatenate(y_train, axis=0)

# Y_train = np.multiply(Y_train,1)
del temp, mask_temp, x, y

In [ ]:
#### Shuffle data

indices = np.arange(x_train.shape[0])
np.random.shuffle(indices)

x_train = x_train[indices, :, :]
y_train = y_train[indices, :, :]

In [ ]:
index = 100
plt.figure(figsize=[10, 10])
plt.subplot(121)
plt.imshow(x_train[index, :, :], cmap="gray")
plt.subplot(122)
plt.imshow(y_train[index, :, :], cmap="gray")
plt.show()

In [ ]:
np.max(x_train)

In [ ]:
np.max(y_train)

In [ ]:
###### Normalize
epsilon = np.finfo(x_train.dtype).eps
for i in range(len(x_train)):
  temp = x_train[i, :, :]
  # temp2 = y_train[i, :, :]

  x_train[i, :, :] = (temp - np.min(temp))/ (np.max(temp)-np.min(temp)+epsilon)
  # y_train[i, :, :] = (temp2 - np.min(temp2))/ (np.max(temp2)-np.min(temp2))



In [ ]:
# from monai.losses import DiceLoss
# import torch.nn as nn

# dice_loss = DiceLoss(sigmoid=True)
# ce_loss = nn.BCELoss()  # For binary segmentation

# def combined_loss(pred, target):
#     return dice_loss(pred, target) + ce_loss(pred, target)




from sklearn.metrics import accuracy_score, precision_score, recall_score



def dice_coef_loss(y_true, y_pred):
    return 1-dice_coef(y_true, y_pred)

def focal_tversky_loss(alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-5):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        tp = tf.reduce_sum(y_true * y_pred)
        fp = tf.reduce_sum(y_pred) - tp
        fn = tf.reduce_sum(y_true) - tp

        tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
        focal_tversky = tf.pow((1 - tversky), gamma)

        return focal_tversky

    return loss

def binary_cross_entropy_loss():
    def loss(y_true, y_pred):
        return tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)

    return loss


def binary_focal_loss(y_true, y_pred, alpha=0.25, gamma=2):
    # Clip the predictions to prevent NaNs
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

    # Calculate focal weight
    focal_weight = alpha * tf.pow(1.0 - y_pred, gamma)

    # Calculate binary cross entropy loss
    bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))

    # Apply focal weight to binary cross entropy   tf.keras.layers.Flatten
    focal_loss = focal_weight * bce

    # Average the loss across batches
    focal_loss = tf.reduce_mean(focal_loss)

    return focal_loss


def dice_coef_loss(y_true, y_pred, smooth=1e-6):
    # Flatten the predictions and labels
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    
    # Calculate the Dice coefficient
    intersection = K.sum(y_true_f * y_pred_f)
    dice = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    
    return dice

def DiceBCELoss(y_true, y_pred, smooth=1e-6):    
    # Cast to float32 data type for compatibility
    y_true = tf.cast(y_true, 'float32')
    y_pred = tf.cast(y_pred, 'float32')
    
    # Flatten the tensors
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    
    # Calculate Binary Cross-Entropy Loss
    BCE = K.binary_crossentropy(y_true_f, y_pred_f)
    
    # Calculate Dice Loss
    dice = dice_coef_loss(y_true_f, y_pred_f)
    dice_loss = 1 - dice
    
    # Combine BCE and Dice Loss
    Dice_BCE = BCE + dice_loss
    
    return Dice_BCE

import tensorflow as tf
import tensorflow.keras.backend as K

# Dice Coefficient Metric
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Sensitivity (Recall) Metric
def sensitivity_metric(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    true_positives = K.sum(y_true_f * y_pred_f)
    possible_positives = K.sum(y_true_f)
    return true_positives / (possible_positives + K.epsilon())

# Precision Metric
def precision_metric(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    true_positives = K.sum(y_true_f * y_pred_f)
    predicted_positives = K.sum(y_pred_f)
    return true_positives / (predicted_positives + K.epsilon())
    
def sensitivity_specific_dice_loss(y_true, y_pred, beta=2, smooth=1):
    # Flatten the tensors
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    
    # Calculate True Positives, False Positives, and False Negatives
    TP = tf.keras.backend.sum(y_true_f * y_pred_f)
    FP = tf.keras.backend.sum(y_pred_f * (1 - y_true_f))
    FN = tf.keras.backend.sum((1 - y_pred_f) * y_true_f)
    
    # Calculate the Recall-Weighted Dice Coefficient
    weighted_dice = (1 + beta) * TP + smooth
    denominator = (1 + beta) * TP + FP + beta * FN + smooth
    dice_loss = 1 - (weighted_dice / denominator)
    
    return dice_loss


def f1_score(y_true, y_pred):
    # Flatten tensors
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    
    # Convert predictions to binary (threshold = 0.5)
    y_pred_f = tf.cast(y_pred_f > 0.5, tf.float32)
    
    # Calculate True Positives, False Positives, and False Negatives
    TP = tf.keras.backend.sum(y_true_f * y_pred_f)
    FP = tf.keras.backend.sum(y_pred_f * (1 - y_true_f))
    FN = tf.keras.backend.sum((1 - y_pred_f) * y_true_f)
    
    # Precision and Recall
    precision = TP / (TP + FP + tf.keras.backend.epsilon())
    recall = TP / (TP + FN + tf.keras.backend.epsilon())
    
    # F1 Score
    f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon())
    return f1

In [ ]:
np.shape(x_train)

In [ ]:
X_train , X_test, Y_train, Y_test = train_test_split(x_train, y_train, test_size=0.05, random_state=32)
del x_train, y_train

In [ ]:
np.shape(X_train)

In [ ]:
X_train = np.reshape(X_train, (len(X_train), 256, 256, 1))
Y_train = np.reshape(Y_train, (len(Y_train), 256, 256, 1))
X_test = np.reshape(X_test, (len(X_test), 256, 256, 1))
Y_test = np.reshape(Y_test, (len(Y_test), 256, 256, 1))

In [ ]:
def gatingsignal(input, out_size, batchnorm=False):
    x = layers.Conv2D(out_size, (1, 1), padding='same')(input)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x
def expend_as(tensor, rep):
     return layers.Lambda(lambda x, repnum: K.repeat_elements(x, repnum, axis=3),
                          arguments={'repnum': rep})(tensor)
def attention_block(x, gating, inter_shape):
    shape_x = K.int_shape(x)
    shape_g = K.int_shape(gating)

    theta_x = layers.Conv2D(inter_shape, (2, 2), strides=(2, 2), padding='same')(x)  # 16
    shape_theta_x = K.int_shape(theta_x)

    phi_g = layers.Conv2D(inter_shape, (1, 1), padding='same')(gating)
    upsample_g = layers.Conv2DTranspose(inter_shape, (3, 3),
                                 strides=(shape_theta_x[1] // shape_g[1], shape_theta_x[2] // shape_g[2]),
                                 padding='same')(phi_g)  # 16

    concat_xg = layers.add([upsample_g, theta_x])
    act_xg = layers.Activation('relu')(concat_xg)
    psi = layers.Conv2D(1, (1, 1), padding='same')(act_xg)
    sigmoid_xg = layers.Activation('sigmoid')(psi)
    shape_sigmoid = K.int_shape(sigmoid_xg)
    upsample_psi = layers.UpSampling2D(size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]))(sigmoid_xg)  # 32

    upsample_psi = expend_as(upsample_psi, shape_x[3])

    y = layers.multiply([upsample_psi, x])

    result = layers.Conv2D(shape_x[3], (1, 1), padding='same')(y)
    result_bn = layers.BatchNormalization()(result)
    return result_bn

def conv_block(x, kernelsize, filters, dropout, batchnorm=False):
    conv = layers.Conv2D(filters, (kernelsize, kernelsize), kernel_initializer='he_normal', padding="same")(x)
    if batchnorm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    if dropout > 0:
        conv = layers.Dropout(dropout)(conv)
    conv = layers.Conv2D(filters, (kernelsize, kernelsize), kernel_initializer='he_normal', padding="same")(conv)
    if batchnorm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    return conv


dropout=0.3
batchnorm=True

filters = [64, 128, 128, 256, 512]
kernelsize = 3
upsample_size = 2
inputs = keras.Input(shape = (256, 256, 1))

# Downsampling layers
dn_1 = conv_block(inputs, kernelsize, filters[0], dropout, batchnorm)
pool_1 = layers.MaxPooling2D(pool_size=(2,2))(dn_1)

dn_2 = conv_block(pool_1, kernelsize, filters[1], dropout, batchnorm)
pool_2 = layers.MaxPooling2D(pool_size=(2,2))(dn_2)

dn_3 = conv_block(pool_2, kernelsize, filters[2], dropout, batchnorm)
pool_3 = layers.MaxPooling2D(pool_size=(2,2))(dn_3)

dn_4 = conv_block(pool_3, kernelsize, filters[3], dropout, batchnorm)
pool_4 = layers.MaxPooling2D(pool_size=(2,2))(dn_4)

dn_5 = conv_block(pool_4, kernelsize, filters[4], dropout, batchnorm)

# Upsampling layers
gating_5 = gatingsignal(dn_5, filters[3], batchnorm)
att_5 = attention_block(dn_4, gating_5, filters[3])
up_5 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(dn_5)
up_5 = layers.concatenate([up_5, att_5], axis=3)
up_conv_5 = conv_block(up_5, kernelsize, filters[3], dropout, batchnorm)

gating_4 = gatingsignal(up_conv_5, filters[2], batchnorm)
att_4 = attention_block(dn_3, gating_4, filters[2])
up_4 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_5)
up_4 = layers.concatenate([up_4, att_4], axis=3)
up_conv_4 = conv_block(up_4, kernelsize, filters[2], dropout, batchnorm)

gating_3 = gatingsignal(up_conv_4, filters[1], batchnorm)
att_3 = attention_block(dn_2, gating_3, filters[1])
up_3 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_4)
up_3 = layers.concatenate([up_3, att_3], axis=3)
up_conv_3 = conv_block(up_3, kernelsize, filters[1], dropout, batchnorm)

gating_2 = gatingsignal(up_conv_3, filters[0], batchnorm)
att_2 = attention_block(dn_1, gating_2, filters[0])
up_2 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_3)
up_2 = layers.concatenate([up_2, att_2], axis=3)
up_conv_2 = conv_block(up_2, kernelsize, filters[0], dropout, batchnorm)

conv_final = layers.Conv2D(1, kernel_size=(1,1))(up_conv_2)
conv_final = layers.BatchNormalization(axis=3)(conv_final)
outputs = layers.Activation('sigmoid')(conv_final)

model = models.Model(inputs=[inputs], outputs=[outputs])
model.summary()



In [ ]:
model.load_weights('/kaggle/working/best_modelAttUnetsecond.h5.keras')

In [ ]:
from keras.callbacks import ModelCheckpoint

filepath = "best_modelAttUnet sen + dice.h5.keras"

checkpoint = ModelCheckpoint(filepath, monitor='val_dice_coef', verbose=1, save_best_only=True, mode='max')

In [ ]:
from sklearn.utils import shuffle
opt = optimizers.Adam(learning_rate = 0.01)

model.compile(optimizer = opt, loss=lambda y_true, y_pred: sensitivity_specific_dice_loss(y_true, y_pred, beta=2) , 
              metrics= [dice_coef, sensitivity_metric, precision_metric, f1_score])


history = model.fit(X_train, Y_train,
                    batch_size=16,
                    shuffle=True,
                    epochs=100,
                    validation_split=0.1,
                    verbose=1,
                    callbacks=[checkpoint])

In [25]:
pred = model.predict(X_test)

2024-11-27 12:50:17.036675: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng15{k5=1,k6=0,k7=1,k10=1} for conv (f32[32,64,256,256]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1,256,256]{3,2,1,0}, f32[64,1,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0} is taking a while...
2024-11-27 12:50:17.118805: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 65536: 1.05936, expected 0.681349
2024-11-27 12:50:17.118845: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 65537: 2.21753, expected 1.83952
2024-11-27 12:50:17.118862: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 65538: 2.5462, expected 2.16819
2024-11-27 12:50:17.118882: E external/local_xla/xla/service/gpu/buffer_comparato

12/13 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step

2024-11-27 12:50:55.844562: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 0: 0.727908, expected 0.233536
2024-11-27 12:50:55.844639: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 1: 1.7588, expected 1.26443
2024-11-27 12:50:55.844656: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 2: 1.59598, expected 1.10161
2024-11-27 12:50:55.844672: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 3: 2.28727, expected 1.7929
2024-11-27 12:50:55.844687: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 4: 2.27906, expected 1.78469
2024-11-27 12:50:55.844706: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 5: 1.91854, expected 1.42417
2024-11-27 12:50:55.844728: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 6: 2.37004, expected 1.87567
2024-11-27 12:50:55.844749: E external/local_xla/xla/se

13/13 ━━━━━━━━━━━━━━━━━━━━ 47s 751ms/step


In [26]:
pred = np.squeeze(pred, axis=-1)

In [29]:
score = model.evaluate(X_test, Y_test, verbose = 0)

print('Test loss:', score[0])
print('Test DICE:', score[1])
print('Test Sen:', score[2])
print('Test Pre:', score[3])
print('Test F1:', score[4])

Test loss: 0.13857795298099518
Test DICE: 0.8696385025978088
Test Sen: 0.8560200333595276
Test Pre: 0.8849055767059326
Test F1: 0.8696752786636353


In [28]:
np.shape(score)

(5,)

In [ ]:
model.save_weights('/kaggle/working/WeightAttUnet-3*100epoch-Lr0.001-S123 data-1-diceloss-bs16-re-tr85va85te88.weights.h5')

In [ ]:
model.save('/kaggle/working/ModelAttUnet-3*100epoch-Lr0.001-S123 data-1-diceloss-bs16-re-tr85va85te88.weights.h5')

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss vs. epochs')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='upper right')
# plt.savefig("2ATTUnet-Dice86loss.png" , dpi = 200)
plt.show()

In [ ]:
plt.plot(history.history['dice_coef'])
plt.plot(history.history['val_dice_coef'])
plt.title('dice_coef vs. epochs')
plt.ylabel('Dice')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='lower right')
# plt.savefig("2ATTUnet-dice86.png" , dpi = 200)
plt.show()

In [ ]:
# model.save_weights('WeightAttUnet-130epoch-Lr0.01-1000 data-diceloss-bs16-re-tr85va82te85.weights.h5')


In [ ]:
pred = model.predict(X_test)

In [ ]:

pred = np.squeeze(pred, axis=-1)

score = model.evaluate(X_test, Y_test, verbose = 0)

print('Test loss:', score[0])
print('Test DICE:', score[1])


In [ ]:

for i in range(len(X_test)):
    plt.figure(figsize=[10, 10])
    plt.subplot(131)
    plt.imshow(pred[i, :, :], cmap="gray")
    plt.subplot(132)
    plt.imshow(Y_test[i, :, :], cmap="gray")
    plt.subplot(133)
    plt.imshow(X_test[i, :, :], cmap="gray")
    plt.savefig("/content/drive/MyDrive/Luna-16/Results Nodule seg/2 Attention U-net/Results P S2F2-6160/Result/"+str(i)+".png")
    plt.show()